# 第 22 章 · FlashAttention：从 FA1 到 FA4，每一代在解决什么瓶颈

本章回答：标准 attention 为什么慢在显存而不是算力？FlashAttention 怎么用分块 + online softmax 做到"不写出 N × N 矩阵"、结果完全精确？FA2、FA3、FA4 分别针对 A100、H100、B200 做了什么改动？

配套讲义：仓库里 `cuda-programming/notes/lessons/22-flashattention.md`（讲解、动画、练习都在讲义里）。对应原文：FlashAttention-2 论文（`materials/text/flashattention-2.txt`：§2.3 第 138–225 行、§3.1 第 234–388 行、§3.2 第 389–469 行）；FlashAttention-3 论文（`text/flashattention-3.txt`：摘要、§3.1 第 208–298 行、§3.2 第 299–369 行、§3.3 FP8 第 370–447 行）；FlashAttention-4 论文（`text/flashattention-4.txt`：摘要、§3.1 第 203–414 行）；GPU MODE L36、L80 字幕；Oxford「FlashAttention」讲义（`text/oxford-FlashAttention.txt`）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 中间矩阵 | 8192 × 8192 × 2 B | `sizes(8192, 128)` | Q/K/V/O 各 2 MB，S/P 各 128 MB，64 倍 |
| 第二步 分块前向 | online softmax 连同 O 打折 | `flash_forward(Q, K, V, 16, 16)` vs `attention(Q, K, V)` | 完全一致（不是近似） |
| 第二步 LSE | $m + \log \ell$ | `flash_forward(...)[1]` | = `log Σ e^S` |
| 第三步 A100 | 312 ÷ 19.5 | `312 / 19.5` | 16 |
| 第五步 FA4 | $4MNd / 8192$、$MN / 16$ | `fa4_cycles(128, 128, 128)` | 1024、1024 周期 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `attention`**（来自 `ch22_flashattention.py`）：标准写法：S = QKᵀ（N × N 整张落地）→ P = softmax(S) → O = PV。这里不除 √d，和讲义的公式一致。

In [ ]:
def attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray) -> np.ndarray:
    """标准写法：S = QKᵀ（N × N 整张落地）→ P = softmax(S) → O = PV。这里不除 √d，和讲义的公式一致。"""
    S = Q @ K.T
    P = np.exp(S - S.max(axis=1, keepdims=True))
    P /= P.sum(axis=1, keepdims=True)
    return P @ V

**函数 `fa4_cycles`**（来自 `ch22_flashattention.py`）：FA4 §3.1.1：一个 tile 里两次 GEMM 共 4MNd FLOP，Tensor Core 每 SM 每周期 8192；exp 每 SM 每周期 16 次。

In [ ]:
def fa4_cycles(m: int, n: int, d: int, mma_flops: int = 8192, exp_per_cycle: int = 16) -> tuple[float, float]:
    """FA4 §3.1.1：一个 tile 里两次 GEMM 共 4MNd FLOP，Tensor Core 每 SM 每周期 8192；exp 每 SM 每周期 16 次。"""
    return 4 * m * n * d / mma_flops, m * n / exp_per_cycle

**函数 `flash_forward`**（来自 `ch22_flashattention.py`）：FA2 前向：每个 Q 块（br 行）扫过所有 K、V 块（bc 行）；S 只在片上存在，用完即丢。返回 O 和 LSE。

In [ ]:
def flash_forward(Q, K, V, br: int, bc: int) -> tuple[np.ndarray, np.ndarray]:
    """FA2 前向：每个 Q 块（br 行）扫过所有 K、V 块（bc 行）；S 只在片上存在，用完即丢。返回 O 和 LSE。"""
    N = Q.shape[0]
    O = np.zeros((N, V.shape[1]))
    LSE = np.zeros(N)
    for i in range(0, N, br):
        q = Q[i:i + br]
        o = np.zeros((len(q), V.shape[1]))
        m = np.full(len(q), -np.inf)
        l = np.zeros(len(q))
        for j in range(0, N, bc):
            S = q @ K[j:j + bc].T                         # br × bc，留在片上
            m_new = np.maximum(m, S.max(axis=1))
            P = np.exp(S - m_new[:, None])
            scale = np.exp(m - m_new)                      # 旧账按新基准打折
            l = scale * l + P.sum(axis=1)
            o = scale[:, None] * o + P @ V[j:j + bc]       # O 也一起打折
            m = m_new
        O[i:i + br] = o / l[:, None]                       # 最后统一除一次 ℓ
        LSE[i:i + br] = m + np.log(l)
    return O, LSE

**常量 `MB`**（来自 `ch22_flashattention.py`）

In [ ]:
MB = 2**20

**函数 `sizes`**（来自 `ch22_flashattention.py`）：一个头：Q / K / V / O 各多大、S / P 各多大（MB）。

In [ ]:
def sizes(n: int, d: int, bytes_per: int = 2) -> tuple[float, float]:
    """一个头：Q / K / V / O 各多大、S / P 各多大（MB）。"""
    return n * d * bytes_per / MB, n * n * bytes_per / MB

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 N = 8192、d = 128、BF16，一个头")
qkv, sp = sizes(8192, 128)
print(f"  Q、K、V、O 各 {qkv:.0f} MB；S、P 各 {sp:.0f} MB；大 {sp / qkv:.0f} 倍")

In [ ]:
print("\n第二步 分块前向 vs 标准 attention（N = 64、d = 16、块 16 × 16）")
rng = np.random.default_rng(0)
Q, K, V = (rng.standard_normal((64, 16)) for _ in range(3))
O, LSE = flash_forward(Q, K, V, 16, 16)
print("  O 一致（不是近似）：", np.allclose(O, attention(Q, K, V)))
S = Q @ K.T
print("  LSE = m + log ℓ 等于 log Σ e^S：", np.allclose(LSE, np.log(np.exp(S).sum(axis=1))))
print("  换成块 8 × 32 结果也一样：", np.allclose(flash_forward(Q, K, V, 8, 32)[0], O))
print(f"  片上一次只放 16 × 16 的 S（{16 * 16} 个数），标准写法要放 64 × 64（{64 * 64} 个）")

In [ ]:
print("\n第三步 A100：FP16 矩阵乘 312 TFLOP/s ÷ 非矩阵乘 FP32 19.5 TFLOP/s =", 312 / 19.5)

In [ ]:
print("\n第五步 FA4 的 feeds and speeds，tile M = N = d = 128")
t_mma, t_exp = fa4_cycles(128, 128, 128)
print(f"  T_MMA = {t_mma:.0f} 周期，T_exp = {t_exp:.0f} 周期")

## 3. 代码里才看得到的

1. **这里没有除以 √d**：讲义第一步写的公式是 $S = QK^\top$，代码照抄；真实 attention 会先乘 $1/\sqrt{d}$，不影响分块算法本身。
2. **换块大小结果不变**：16 × 16 和 8 × 32 算出同一个 O——分块只改变"一次在片上放多少"，不改变数学结果。
3. **`attention` 会把整张 N × N 的 S 放进内存**：N = 64 时只有 4096 个数；讲义的 N = 8192 时是 128 MB，这就是 FlashAttention 要解决的问题。

## 4. 自己改一改

- 把块大小换成 64 × 64（一次放下整张 S），看 `flash_forward` 是不是退化成了标准写法。
- 把 Q、K 乘 10 让分数变大，确认分块版仍然不溢出、和标准版一致。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 标准 attention 慢在 N 乘 N：中间矩阵比输入大几十倍，还要写回显存；
> FlashAttention：分块搬进片上，online softmax 边算边改账，连 O 一起打折，最后除一次 ℓ；不近似、不落地、反向重算；
> FA2 晚除 ℓ、按 Q 再并行；FA3 搬算分工、乒乓重叠、上 FP8；FA4 发现 exp 跟不上，多项式顶上、能不 rescale 就不 rescale；
> 每一代硬件换瓶颈，算法跟着换。